In [14]:
import requests
import pandas as pd
import time
import os
import json

In [50]:
# Download Data
# the results will be stored in the folder "data"
# the folder structure should be:
# project-chaggg/
# ├── data/
# │   └── chicago_crime_data/
# │       ├── batches/
# │       └── chicago_crimes_2001_2025.csv

# ── Config ────────────────────────────────────────────────────────────────────
APP_TOKEN    = "hRLoOGqjoDwIbBJGB949EQ2AZ"
BASE_URL     = "https://data.cityofchicago.org/resource/ijzp-q8t2.json"
BATCH_SIZE   = 10000
BASE_DIR = os.getcwd()
OUTPUT_DIR   = os.path.join(BASE_DIR, "data", "chicago_crime_data")
FINAL_OUTPUT = os.path.join(OUTPUT_DIR, "chicago_crimes_2001_2025.csv")
PROGRESS_FILE = os.path.join(OUTPUT_DIR, "progress.json")
BATCH_DIR = os.path.join(OUTPUT_DIR, "batches") # this line was not in the last version
os.makedirs(BATCH_DIR, exist_ok=True)

EXPECTED_COLUMNS = [
    'id', 'case_number', 'date', 'block', 'iucr', 'primary_type',
    'description', 'location_description', 'arrest', 'domestic', 'beat',
    'district', 'ward', 'community_area', 'fbi_code', 'year', 'updated_on',
    'x_coordinate', 'y_coordinate', 'latitude', 'longitude', 'location'
]

# ── Progress helpers ──────────────────────────────────────────────────────────
def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f:
            return json.load(f)
    return {"offset": 0, "batch_num": 1, "total_records": 0}

def save_progress(offset, batch_num, total):
    with open(PROGRESS_FILE, "w") as f:
        json.dump({"offset": offset, "batch_num": batch_num, "total_records": total}, f)

# ── Fetch ─────────────────────────────────────────────────────────────────────
def fetch_batch(offset, retries=3):
    params = {
        "$limit":      BATCH_SIZE,
        "$offset":     offset,
        "$order":      "date ASC",
        "$$app_token": APP_TOKEN
    }
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(BASE_URL, params=params, timeout=60)
            r.raise_for_status()
            return r.json()
        except requests.RequestException as e:
            print(f"  [Attempt {attempt}/{retries}] Failed: {e}")
            if attempt < retries:
                time.sleep(5 * attempt)
            else:
                raise

# ── Align columns ─────────────────────────────────────────────────────────────
def align_columns(df):
    for col in EXPECTED_COLUMNS:
        if col not in df.columns:
            df[col] = None
    return df[EXPECTED_COLUMNS]

# ── Download ──────────────────────────────────────────────────────────────────
def download_all():
    if os.path.exists(FINAL_OUTPUT):
        print(f"File already exists: {FINAL_OUTPUT}\nDownload skipped.")
        return
    progress  = load_progress()
    offset    = progress["offset"]
    batch_num = progress["batch_num"]
    total     = progress["total_records"]

    print(f"{'Resuming' if offset > 0 else 'Starting'} download "
          f"(batch={batch_num}, offset={offset}, saved={total})\n")

    while True:
        batch_file = os.path.join(BATCH_DIR, f"batch_{batch_num:04d}.csv")

        if os.path.exists(batch_file):
            print(f"Batch {batch_num} already exists, skipping...")
            offset    += BATCH_SIZE
            batch_num += 1
            continue

        print(f"Fetching batch {batch_num} (offset={offset})...")
        batch = fetch_batch(offset)

        if not batch:
            print("No more data. Download complete.")
            break

        df = align_columns(pd.DataFrame(batch))
        df.to_csv(batch_file, index=False)

        total     += len(batch)
        offset    += BATCH_SIZE
        batch_num += 1
        save_progress(offset, batch_num, total)
        print(f"  -> {len(batch)} records. Total: {total}")
        time.sleep(0.5)

    if os.path.exists(PROGRESS_FILE):
        os.remove(PROGRESS_FILE)

# ── Stream-merge batches ──────────────────────────────────────────────────────
def combine_batches():
    if os.path.exists(FINAL_OUTPUT):
        print(f"Final file already exists: {FINAL_OUTPUT}, skipping merge.")
        return

    batch_files = sorted(
        os.path.join(BATCH_DIR, f)
        for f in os.listdir(BATCH_DIR) if f.endswith(".csv")
    )
    print(f"\nMerging {len(batch_files)} batches into final CSV...")

    with open(FINAL_OUTPUT, "w", encoding="utf-8") as out:
        for i, path in enumerate(batch_files):
            df = pd.read_csv(path, low_memory=False)
            df = align_columns(df)                      # re-align just in case
            df.to_csv(out, index=False, header=(i == 0))
            print(f"  Merged {path} ({len(df)} rows)")

    print(f"\nDone! Saved to: {FINAL_OUTPUT}")

# ── Entry point ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    download_all()
    combine_batches()

Resuming download (batch=854, offset=8530000, saved=8530000)

Fetching batch 854 (offset=8530000)...
  -> 1378 records. Total: 8531378
Fetching batch 855 (offset=8540000)...
No more data. Download complete.

Merging 854 batches into final CSV...
  Merged /Users/gracewang/Documents/Hertie/Data Structures & Algorithms/project-chaggg/notebooks/data/chicago_crime_data/batches/batch_0001.csv (10000 rows)
  Merged /Users/gracewang/Documents/Hertie/Data Structures & Algorithms/project-chaggg/notebooks/data/chicago_crime_data/batches/batch_0002.csv (10000 rows)
  Merged /Users/gracewang/Documents/Hertie/Data Structures & Algorithms/project-chaggg/notebooks/data/chicago_crime_data/batches/batch_0003.csv (10000 rows)
  Merged /Users/gracewang/Documents/Hertie/Data Structures & Algorithms/project-chaggg/notebooks/data/chicago_crime_data/batches/batch_0004.csv (10000 rows)
  Merged /Users/gracewang/Documents/Hertie/Data Structures & Algorithms/project-chaggg/notebooks/data/chicago_crime_data/batch

In [56]:
# Load the data

df = pd.read_csv("data/chicago_crime_data/chicago_crimes_2001_2025.csv", low_memory = False)

FileNotFoundError: [Errno 2] No such file or directory: 'notebooks/data/chicago_crime_data/chicago_crimes_2001_2025.csv'

In [52]:
# Check the basic info of data

print(df.describe())
print(df.shape)          # rows and columns
print(df.dtypes)         # data types
print(df.columns.tolist()) # column names

                 id          beat      district          ward  community_area  \
count  8.531378e+06  8.531378e+06  8.531331e+06  7.916562e+06    7.917651e+06   
mean   7.599243e+06  1.183071e+03  1.129542e+01  2.278825e+01    3.736368e+01   
std    3.828745e+06  7.038039e+02  6.965142e+00  1.386001e+01    2.154627e+01   
min    6.340000e+02  1.110000e+02  1.000000e+00  1.000000e+00    0.000000e+00   
25%    4.118663e+06  6.210000e+02  6.000000e+00  1.000000e+01    2.300000e+01   
50%    7.597312e+06  1.034000e+03  1.000000e+01  2.300000e+01    3.200000e+01   
75%    1.103141e+07  1.731000e+03  1.700000e+01  3.400000e+01    5.600000e+01   
max    1.416547e+07  2.535000e+03  3.100000e+01  5.000000e+01    7.700000e+01   

               year  x_coordinate  y_coordinate      latitude     longitude  
count  8.531378e+06  8.435045e+06  8.435045e+06  8.435045e+06  8.435045e+06  
mean   2.011189e+03  1.164670e+06  1.885930e+06  4.184259e+01 -8.767124e+01  
std    7.212996e+00  1.694365e+04  3

In [53]:
# Filter the data from 2001 to 2025

df = pd.read_csv(FINAL_OUTPUT, low_memory=False)
df = df[df['year'].between(2001, 2025)]
df.to_csv(FINAL_OUTPUT, index=False)
print(df['year'].value_counts().sort_index())
print(f"\nTotal records (2001-2025): {len(df)}")

year
2001    485963
2002    486832
2003    476000
2004    469443
2005    453789
2006    448200
2007    437109
2008    427218
2009    392866
2010    370568
2011    352059
2012    336376
2013    307620
2014    275881
2015    264896
2016    269972
2017    269292
2018    269157
2019    261710
2020    212725
2021    209682
2022    240035
2023    263313
2024    259074
2025    236972
Name: count, dtype: int64

Total records (2001-2025): 8476752


In [54]:
# Check missing data

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
}).sort_values("missing_pct", ascending=False)

missing_summary

,missing_count,missing_pct
ward,614816,7.25
community_area,613726,7.24
location,96251,1.14
longitude,96251,1.14
latitude,96251,1.14
y_coordinate,96251,1.14
x_coordinate,96251,1.14
location_description,15613,0.18
updated_on,0,0.00
year,0,0.00


## Column Alignment Check

All 22 columns are present and correctly ordered — no misalignment detected.

Missing values are due to **genuine data gaps**, not structural issues:

| Column | Missing | Reason |
|--------|---------|--------|
| `ward` / `community_area` | ~614k | Early records do not include these fields |
| `x/y_coordinate` / `latitude` / `longitude` / `location` | ~94k | Addresses that could not be geocoded |
| `location_description` | ~15k | Small number of records with no venue description |
| `district` | 47 | Negligible |